# Agente ReAct com memória — base revisada

Integração exploratória do **AI Investigation & Executive Copilot**, com ferramentas compartilhadas com a **Control Tower** e o **Order Margin Engine**. Revisão16/09/2026. Não é ainda a interface final do produto.

As saídas antigas foram retiradas porque continham cálculos e conclusões supersedidos. Executar não chama a API por padrão. Fonte técnica: `../solucao/ESPECIFICACAO_PROTOTIPO.md`.


## 1 · ReAct com memória no contexto do produto

ReAct permite alternar consulta e observação quando o recorte seguinte depende do resultado anterior. Essa é uma possibilidade de investigação controlada, não uma exigência de autonomia ampla. Na primeira onda do produto, priorizar consulta, explicação e memo sobre métricas governadas.

A memória mantém definições e contexto revisável; ela não prova causalidade e não impede reabrir uma hipótese quando surge evidência nova. As memórias anteriores com alegações sobre frete fixo, ausência de efeito de canal e margem realizada foram retiradas do contexto ativo.

Fluxo demonstrável: pergunta → ferramenta autorizada → resultado com escopo e evidence_id → explicação com limitações → revisão humana. Código determinístico reduz erros de cálculo, mas não garante interpretação factual do modelo.


## 2 · Configuração

Use o ambiente Python existente. Para executar consultas reais, configure `VERTICE_API_KEY` no ambiente e marque `EXECUTAR_LLM=True`. Sem chave, a verificação determinística funciona. A chave não é salva no notebook. O nome de modelo é configurável; disponibilidade depende do provedor.


In [1]:
# Dependências já disponíveis no ambiente .venv; nenhuma instalação automática.
# pandas, langchain-core, langchain-litellm, langgraph, IPython.


In [2]:
import os, json, sys
from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_litellm import ChatLiteLLM
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver
API_KEY = os.environ.get("VERTICE_API_KEY", "")
API_BASE_URL = os.environ.get("VERTICE_API_BASE_URL", "https://chat.eloagents.click/api")
MODELO = os.environ.get("VERTICE_MODEL", "claude-opus-47")
EXECUTAR_LLM = False
TEM_CHAVE = bool(API_KEY)
llm = None
print("Chamadas ao modelo desabilitadas por padrão; ferramentas locais disponíveis.")


Chamadas ao modelo desabilitadas por padrão; ferramentas locais disponíveis.


## 3 · Fundação compartilhada

O notebook importa o mesmo cálculo usado pelas futuras telas e tools. Reexecução requer a pasta `prototipo/`, seu contrato e `data-room/`; não é mais um arquivo autossuficiente isolado.


In [3]:
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "prototipo/core.py").is_file())
if str(RAIZ) not in sys.path: sys.path.insert(0,str(RAIZ))
from prototipo.core import DataRoom, Scope
from prototipo.copilot import ToolSession, SYSTEM_PROMPT
room = DataRoom(RAIZ)
print(room.metrics()["valores"])


{'pedidos': 19903, 'receita_bruta': 14754769.54, 'desconto_reais': 1180841.92, 'receita_liquida': 13573927.62, 'custo_produto': 5949986.35, 'custo_frete': 241040.26, 'margem_contribuicao': 7382901.01, 'mc_pct': 54.390307777403635, 'desconto_pct_bruta': 8.003120054154367, 'pedidos_B_no_recorte': 19903, 'participacao_B_pct': 100.0, 'pedidos_com_atendimento': 6272, 'tickets_vinculados': 7680, 'cobertura_atendimento_pct': 31.512837260714466, 'atendimento_vinculado_registrado': 113434.0, 'custo_vinculado_por_pedido_populacao': 5.699341807767674, 'janela_tickets_vinculados': ['2023-01-01 08:32:11', '2025-12-31 18:22:03']}


## 4 · Ferramentas autorizadas

Os contratos e filtros são explícitos. Retornos carregam IDs de evidência, versão, população e fontes. Funil não calcula margem realizada; estratificação não é teste causal. Não há ferramenta de execução livre de código ou publicação.


In [4]:
@tool
def metricas(scope: dict = {}) -> str:
    """Consulta KPIs governados. Scope: start/end,population A/B,channel,category,ticket_lt,discounted_only. Padrão B/2023."""
    return json.dumps(room.metrics(Scope(**scope)),ensure_ascii=False)
@tool
def decompor(dimension: str, scope: dict = {}) -> str:
    """Decompõe por canal,categoria,metodo_pagamento,mes,com_desc,com_frete,faixa_ticket. Não infere causas."""
    return json.dumps(room.breakdown(dimension,Scope(**scope)),ensure_ascii=False)
@tool
def funil(scope: dict = {"population":"A"}) -> str:
    """Conta grupos exclusivos de status, população A. MC registrada não representa perda realizada."""
    return json.dumps(room.funnel(Scope(**scope)),ensure_ascii=False)
@tool
def simular_desconto(scope: dict = {"ticket_lt":250,"discounted_only":True}, removal_fraction: float = 1.0, retention: float = 1.0) -> str:
    """Cenário aritmético de desconto. Premissas entre0 e1; retenção não é previsão. M/(M+D)."""
    return json.dumps(room.simulate_discount(Scope(**scope),removal_fraction,retention),ensure_ascii=False)
@tool
def comparar_estratos(dimension: str, control: str, scope: dict = {}) -> str:
    """Comparação descritiva estratificada; não identifica causalidade."""
    return json.dumps(room.stratified(dimension,control,Scope(**scope)),ensure_ascii=False)
FERRAMENTAS = [metricas,decompor,funil,simular_desconto,comparar_estratos]


In [5]:
# As ferramentas do grafo serão executadas por ToolSession com orçamento por missão.
# Não há cálculo financeiro alternativo nas células de integração.


### Memória revisada

Curto prazo: histórico por thread, com evidência renovada por missão. Semântica: contexto e limites revisados. Episódica: apenas registros ativos curados, sem importar memórias históricas supersedidas. Novos achados devem ser registrados por humano no futuro Decision Log; o agente não promove descobertas automaticamente.


In [6]:
def carregar_memoria():
    semantic = (RAIZ / "agente/memoria_semantica.md").read_text()
    records = json.loads((RAIZ / "agente/memoria_episodica.json").read_text())
    today = pd.Timestamp.today().date().isoformat()
    active = [r for r in records if r.get("status") in {"confirmado","aberto"} and r.get("valido_ate","")>=today]
    return semantic + "\nContexto episódico (não substitui tools):\n" + json.dumps(active,ensure_ascii=False)


## 5 · Verificação sem API

Estes asserts verificam regressões financeiras. A suíte completa fica em `tests/test_foundation.py`. Não avaliam factualidade de um modelo real.


In [7]:
sessao_teste = ToolSession(room)
a = sessao_teste.call("metricas")
assert a["valores"]["pedidos"] == 19903
assert a["valores"]["margem_contribuicao"] == 7382901.01
s = sessao_teste.call("simular_desconto",{"scope":{"ticket_lt":250,"discounted_only":True}})
assert abs(s["valores"]["retencao_equilibrio"] - .4362356013832186)<1e-10
z = sessao_teste.call("simular_desconto",{"scope":{"ticket_lt":250,"discounted_only":True},"retention":0})
assert z["valores"]["delta_mc_cenario"] == -73913.06
assert sessao_teste.check_references([s["evidence_id"]])["referencias_validas"]
print("Verificação financeira passou. Nenhuma chamada de API executada.")


Verificação financeira passou. Nenhuma chamada de API executada.


## 6 · ReAct limitado com memória

Toda missão cria orçamento e trilha próprios. Erros não contam como evidência. Ao esgotar o orçamento, retorna evidências e limitações sem forçar uma conclusão do LLM. A existência de evidências ainda não garante que toda frase gerada seja fiel; avaliar antes de uso executivo.


In [8]:
SISTEMA = SYSTEM_PROMPT
MAX_TOOLS = 8


### Grafo

`início → agente → ferramentas governadas → agente → fim`

Estado mantém histórico de mensagens e uma ToolSession nova a cada missão. Não executa aprovação comercial nem grava descobertas automaticamente.


In [9]:
class EstadoAgente(MessagesState):
    trace: list
    evidencias: dict

def no_agente(estado):
    if len(estado["trace"])>=MAX_TOOLS:
        from langchain_core.messages import AIMessage
        ids=list(estado["evidencias"])
        return {"messages":[AIMessage(content="Limite de ferramentas atingido. Evidências disponíveis: " + ", ".join(ids) + ". Síntese não concluída; consultar os resultados registrados e revisar antes de decidir.")]}
    return {"messages":[llm.bind_tools(FERRAMENTAS).invoke([SystemMessage(content=SISTEMA+"\n"+carregar_memoria()),*estado["messages"]])]}

def no_tools(estado):
    outputs=[]
    session=ToolSession(room,max_calls=MAX_TOOLS)
    session.trace=list(estado["trace"])
    session.results=dict(estado["evidencias"])
    for call in estado["messages"][-1].tool_calls:
        args=dict(call["args"])
        # Apply documented defaults consistently to direct and session-based tools.
        if call["name"]=="funil": args.setdefault("scope",{"population":"A"})
        if call["name"]=="simular_desconto":args.setdefault("scope",{"ticket_lt":250,"discounted_only":True})
        result=session.call(call["name"],args)
        outputs.append(ToolMessage(content=json.dumps(result,ensure_ascii=False),tool_call_id=call["id"]))
    return {"messages":outputs,"trace":session.trace,"evidencias":session.results}

def route(estado):
    return "tools" if getattr(estado["messages"][-1],"tool_calls",None) else END

builder=StateGraph(EstadoAgente)
builder.add_node("agente",no_agente);builder.add_node("tools",no_tools)
builder.add_edge(START,"agente");builder.add_conditional_edges("agente",route,{"tools":"tools",END:END});builder.add_edge("tools","agente")
grafo=builder.compile(checkpointer=InMemorySaver())
print("Grafo compilado. Sem execução de modelo.")


Grafo compilado. Sem execução de modelo.


## 7 · Missões opcionais

A chave vem do ambiente e a execução precisa ser ativada na configuração. Falha de API não deve gerar uma resposta factual fictícia.


In [10]:
def missao(pergunta,thread="vertice-01"):
    global llm
    if not EXECUTAR_LLM or not TEM_CHAVE:
        print("Execução de IA desabilitada. Configure VERTICE_API_KEY e EXECUTAR_LLM para uma rodada real.")
        return None
    if llm is None:
        llm=ChatLiteLLM(model="openai/"+MODELO,api_key=API_KEY,api_base=API_BASE_URL,request_timeout=90)
    result=grafo.invoke({"messages":[HumanMessage(content=pergunta)],"trace":[],"evidencias":{}},config={"configurable":{"thread_id":thread},"recursion_limit":MAX_TOOLS*2+6})
    if not result["evidencias"]:
        display(Markdown("**Abstenção:** nenhuma evidência válida foi consultada nesta missão. Resposta não liberada."))
    else:
        display(Markdown("**Rascunho para revisão; factualidade não certificada automaticamente.**\n\n"+str(result["messages"][-1].content)))
    return {"rascunho":str(result["messages"][-1].content) if result["evidencias"] else None,"rastro":{"chamadas":result["trace"],"evidencias":result["evidencias"]}}


### Missão1 — descrição com escopo


In [11]:
r1 = missao("Compare a MC observável de B/2023 por canal. Cite evidências e limites.")


Execução de IA desabilitada. Configure VERTICE_API_KEY e EXECUTAR_LLM para uma rodada real.


### Missão2 — cenário, não previsão


In [12]:
r2 = missao("Simule remover metade do desconto em B/2023, ticket abaixo de250, com desconto. Assuma retenção0.8 e mostre as premissas.")


Execução de IA desabilitada. Configure VERTICE_API_KEY e EXECUTAR_LLM para uma rodada real.


### Missão3 — pergunta não identificável


In [13]:
r3 = missao("Quanto lucro líquido recuperaremos com devoluções? Se não for calculável, explique os dados faltantes.",thread="vertice-limites")


Execução de IA desabilitada. Configure VERTICE_API_KEY e EXECUTAR_LLM para uma rodada real.


## 8 · Curadoria

Evidências produzidas são propostas para análise humana. A versão atual não contém função para promover uma conclusão automaticamente. A persistência/revisão do produto será implementada no Decision Log.


In [14]:
display(pd.DataFrame(json.loads((RAIZ / "agente/memoria_episodica.json").read_text()))[["id","titulo","status"]])


,id,titulo,status
0,REV-001,Populações da análise,confirmado
1,REV-002,Frete é componente contábil,aberto
2,REV-003,Simulação de desconto,aberto
3,REV-004,Pós-venda e devoluções,confirmado
4,REV-005,Atendimento vinculado,confirmado
5,REV-006,Estoque e aquisição,confirmado


In [15]:
print("Memórias anteriores preservadas em _arquivo/revisao_pre_prototipo; não são contexto ativo.")


Memórias anteriores preservadas em _arquivo/revisao_pre_prototipo; não são contexto ativo.


## 9 · Próximas avaliações

Testar perguntas respondíveis, abstenções, filtros inválidos, orçamento, indisponibilidade, referências inventadas e confusão entre associação e causa. Comparar resultados com os contratos e revisão humana.


In [16]:
# Não executar novas missões automaticamente; não há avaliação LLM nesta revisão.


## 10 · Limites da implementação

A fundação e os testes determinísticos não certificam factualidade do modelo. A interface, o Decision Log persistente, o conjunto de avaliação e a autenticação de produto ainda precisam ser construídos. Memória de thread é local ao kernel; não é armazenamento durável de decisões.
